# Notebook 03 — Encoded Pairs & Data Quality (Stage 6)

Deep inspection of the integer-encoded context-response pairs produced by Stage 6.

| Section | Topic |
|---------|-------|
| **C** | Pipeline config, inventory, disk sizes, config hashes |
| **D** | Sample pair inspection & decoding |
| **E** | Sequence length statistics and histograms |
| **F** | Token distribution & data quality metrics |
| **G** | Split integrity checks (leakage, ID range, SOS/EOS, __eot__) |
| **H** | Summary dashboard |


In [ ]:
# === SETUP: Imports, Paths, and Helpers ===

import json
import hashlib
import math
import os
import random
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
import sentencepiece as spm
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
VIZ_DIR      = PROJECT_ROOT / 'notebooks' / 'visualizations'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL  = ARTIFACT_DIR / 'stage6_train_ids.jsonl'
VAL_JSONL    = ARTIFACT_DIR / 'stage6_val_ids.jsonl'
TEST_JSONL   = ARTIFACT_DIR / 'stage6_test_ids.jsonl'
VOCAB_JSON   = ARTIFACT_DIR / 'stage6_vocab.json'
IDX2WORD_JSON= ARTIFACT_DIR / 'stage6_idx2word.json'
STAGE6_STATS = ARTIFACT_DIR / 'stage6_stats.json'
SPM_MODEL    = ARTIFACT_DIR / 'stage5_spm.model'

# Config constants from phase1.py
MAX_CTX_TOKENS  = 256
MAX_RESP_TOKENS = 50
PAD_ID, UNK_ID, SOS_ID, EOS_ID = 0, 1, 2, 3

plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11,
})
sns.set_palette('muted')
console = Console(highlight=True)
random.seed(42)

def _require(path: Path, label: str = '') -> bool:
    if path.exists():
        return True
    console.print(Panel(
        f'[bold red]File not found:[/bold red] {path}\n'
        f'[yellow]Run phase1.py to generate {label or path.name}.[/yellow]',
        title='⚠  Missing Artefact', border_style='red'
    ))
    return False

def _load_jsonl_full(path: Path) -> list:
    recs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            recs.append(json.loads(line))
    return recs

def _save_fig(fig, name: str):
    out = VIZ_DIR / f'{name}.png'
    fig.savefig(out, bbox_inches='tight', dpi=150)
    console.print(f'[green]Saved[/green] → {out}')

console.print(Panel('[bold green]Setup complete[/bold green]\n'
    f'ARTIFACT_DIR = {ARTIFACT_DIR}\n'
    f'MAX_CTX_TOKENS={MAX_CTX_TOKENS}  MAX_RESP_TOKENS={MAX_RESP_TOKENS}  (phase1 config)',
    title='Notebook 03 — Encoded Pairs & Data Quality', border_style='green'))


## Section C — Pipeline Config & Inventory

In [ ]:
# === SECTION C.1: stage6_stats.json ===

if not _require(STAGE6_STATS, 'stage6_stats.json'):
    raise SystemExit('Missing stage6 stats')

with open(STAGE6_STATS) as f:
    s6 = json.load(f)

console.print(Panel(json.dumps(s6, indent=2), title='stage6_stats.json', border_style='blue'))

VOCAB_SIZE = s6.get('vocab_size', 32000)
N_TRAIN    = s6.get('n_train', 0)
N_VAL      = s6.get('n_val', 0)
N_TEST     = s6.get('n_test', 0)

t = Table(title='Stage 6 Key Config', box=box.SIMPLE_HEAVY)
for k, v in [('vocab_size', VOCAB_SIZE), ('n_train', N_TRAIN),
              ('n_val', N_VAL), ('n_test', N_TEST),
              ('sos_id', s6.get('sos_id', 2)), ('eos_id', s6.get('eos_id', 3))]:
    t.add_row(k, str(v))
console.print(t)


In [ ]:
# === SECTION C.2: Line counts table for all 3 JSONL splits ===

N_TOTAL = N_TRAIN + N_VAL + N_TEST
t = Table(title='JSONL Line Counts vs Expected Split Ratios', box=box.ROUNDED)
t.add_column('Split', style='cyan')
t.add_column('Expected ~', justify='right')
t.add_column('Actual Lines', justify='right')
t.add_column('Actual %', justify='right')
t.add_column('Status', justify='center')

EXPECTED = {'train': 0.80, 'val': 0.10, 'test': 0.10}
TOLERANCE = 0.05

actual_counts = {}
for split_name, fpath, n_stat in [
    ('train', TRAIN_JSONL, N_TRAIN), ('val', VAL_JSONL, N_VAL), ('test', TEST_JSONL, N_TEST)
]:
    if not fpath.exists():
        t.add_row(split_name, '—', 'MISSING', '—', '[red]MISSING[/red]')
        actual_counts[split_name] = 0
        continue
    with open(fpath) as f:
        actual_n = sum(1 for _ in f)
    actual_counts[split_name] = actual_n
    actual_pct = actual_n / N_TOTAL if N_TOTAL > 0 else 0
    exp_pct = EXPECTED[split_name]
    ok = abs(actual_pct - exp_pct) <= TOLERANCE
    status = f'[green]OK[/green]' if ok else f'[red]WARN[/red]'
    t.add_row(split_name, f'~{exp_pct:.0%}', f'{actual_n:,}', f'{actual_pct:.1%}', status)
console.print(t)


In [ ]:
# === SECTION C.3: Disk size table for stage 6 artifacts ===

t = Table(title='Stage 6 Artifact Disk Sizes', box=box.ROUNDED)
t.add_column('File', style='cyan')
t.add_column('Exists', justify='center')
t.add_column('Size (MB)', justify='right')

stage6_files = [
    ('stage6_train_ids.jsonl', TRAIN_JSONL),
    ('stage6_val_ids.jsonl',   VAL_JSONL),
    ('stage6_test_ids.jsonl',  TEST_JSONL),
    ('stage6_vocab.json',      VOCAB_JSON),
    ('stage6_idx2word.json',   IDX2WORD_JSON),
    ('stage6_stats.json',      STAGE6_STATS),
]
for fname, fpath in stage6_files:
    exists = fpath.exists()
    size_mb = f'{fpath.stat().st_size / 1e6:.2f}' if exists else '—'
    t.add_row(fname, '[green]✓[/green]' if exists else '[red]✗[/red]', size_mb)
console.print(t)


In [ ]:
# === SECTION C.4: Config hash stamps for stages 1-8 ===

t = Table(title='Config Hash Stamps (.stageN_config_hash)', box=box.ROUNDED)
t.add_column('Stage', style='cyan')
t.add_column('File', style='dim')
t.add_column('Hash', style='yellow')
t.add_column('Exists', justify='center')

for n in range(1, 9):
    hf = ARTIFACT_DIR / f'.stage{n}_config_hash'
    exists = hf.exists()
    hashval = hf.read_text().strip() if exists else '—'
    t.add_row(f'Stage {n}', hf.name,
              hashval[:40] if hashval != '—' else '—',
              '[green]✓[/green]' if exists else '[yellow]—[/yellow]')
console.print(t)


## Section D — Sample Pair Inspection

In [ ]:
# === SECTION D.1: Load vocab + SPM for decoding ===

if not (_require(VOCAB_JSON, 'stage6_vocab.json') and
        _require(IDX2WORD_JSON, 'stage6_idx2word.json') and
        _require(SPM_MODEL, 'stage5_spm.model')):
    raise SystemExit('Missing vocab/SPM artifacts')

with open(VOCAB_JSON) as f:
    vocab = json.load(f)
with open(IDX2WORD_JSON) as f:
    idx2word = json.load(f)

sp = spm.SentencePieceProcessor()
sp.Load(str(SPM_MODEL))

EOT_ID = sp.piece_to_id('__eot__')
console.print(f'Vocab size: {len(vocab)}  |  SPM pieces: {sp.GetPieceSize()}  |  __eot__ id: {EOT_ID}')


In [ ]:
# === SECTION D.2: decode_sequence helper ===

def decode_ctx(ids: list) -> str:
    """Decode context IDs. __eot__ shown as [__eot__], other specials decoded naturally."""
    parts = []
    current_segment = []
    for tid in ids:
        if tid == EOT_ID:
            if current_segment:
                try:
                    parts.append(sp.decode(current_segment))
                except Exception:
                    parts.append(str(current_segment))
                current_segment = []
            parts.append('[__eot__]')
        elif tid in (PAD_ID, SOS_ID, EOS_ID):
            if current_segment:
                try:
                    parts.append(sp.decode(current_segment))
                except Exception:
                    parts.append(str(current_segment))
                current_segment = []
            parts.append(f'[{idx2word.get(str(tid), f"id={tid}")}]')
        else:
            current_segment.append(tid)
    if current_segment:
        try:
            parts.append(sp.decode(current_segment))
        except Exception:
            parts.append(str(current_segment))
    return ' '.join(p for p in parts if p)

def decode_resp(ids: list) -> str:
    """Decode response IDs: strip leading SOS and trailing EOS, then decode."""
    inner = ids
    if inner and inner[0] == SOS_ID:
        inner = inner[1:]
    if inner and inner[-1] == EOS_ID:
        inner = inner[:-1]
    decodable = [tid for tid in inner if tid not in (PAD_ID,)]
    if not decodable:
        return '[empty]'
    try:
        return sp.decode(decodable)
    except Exception:
        return str(decodable[:20])

console.print('[bold green]decode_ctx() and decode_resp() helpers defined.[/bold green]')


In [ ]:
# === SECTION D.3: Load all 3 splits; print 10 random train samples ===

console.print('[bold]Loading JSONL splits (this may take a moment for large files)...[/bold]')
train_data = _load_jsonl_full(TRAIN_JSONL) if TRAIN_JSONL.exists() else []
val_data   = _load_jsonl_full(VAL_JSONL)   if VAL_JSONL.exists()   else []
test_data  = _load_jsonl_full(TEST_JSONL)  if TEST_JSONL.exists()  else []
console.print(f'Loaded: train={len(train_data):,}  val={len(val_data):,}  test={len(test_data):,}')

sample_train = random.sample(train_data, min(10, len(train_data)))
console.print('[bold]\n10 Random Train Samples:[/bold]')
for i, rec in enumerate(sample_train, 1):
    ctx_ids  = rec.get('ctx', [])
    resp_ids = rec.get('resp', [])
    ctx_txt  = decode_ctx(ctx_ids)
    resp_txt = decode_resp(resp_ids)
    console.print(Panel(
        f'[bold cyan]CTX[/bold cyan]  (len={len(ctx_ids)}): {ctx_txt[:300]}\n'
        f'[bold green]RESP[/bold green] (len={len(resp_ids)}): {resp_txt[:200]}',
        title=f'Train Sample {i}/10', border_style='blue', padding=(0, 1)
    ))


In [ ]:
# === SECTION D.4: 5 val + 5 test samples ===

for split_name, split_data in [('Val', val_data), ('Test', test_data)]:
    sample = random.sample(split_data, min(5, len(split_data)))
    for i, rec in enumerate(sample, 1):
        ctx_ids  = rec.get('ctx', [])
        resp_ids = rec.get('resp', [])
        console.print(Panel(
            f'[bold cyan]CTX[/bold cyan]  (len={len(ctx_ids)}): {decode_ctx(ctx_ids)[:280]}\n'
            f'[bold green]RESP[/bold green] (len={len(resp_ids)}): {decode_resp(resp_ids)[:180]}',
            title=f'{split_name} Sample {i}/5', border_style='magenta', padding=(0, 1)
        ))


In [ ]:
# === SECTION D.5: Flag suspicious pairs ===

def _check_suspicious(data, split_name):
    n_ctx_empty      = 0
    n_resp_empty     = 0
    n_resp_spec_only = 0
    n_resp_no_eos    = 0
    SPECIAL_ONLY_IDS = {PAD_ID, UNK_ID, SOS_ID, EOS_ID}
    for rec in data:
        ctx  = rec.get('ctx', [])
        resp = rec.get('resp', [])
        if not ctx:
            n_ctx_empty += 1
        if not resp or all(tid in SPECIAL_ONLY_IDS for tid in resp):
            n_resp_empty += 1
        if resp and all(tid in SPECIAL_ONLY_IDS for tid in resp):
            n_resp_spec_only += 1
        if resp and resp[-1] != EOS_ID:
            n_resp_no_eos += 1
    return n_ctx_empty, n_resp_empty, n_resp_spec_only, n_resp_no_eos

t = Table(title='Suspicious Pair Flags', box=box.ROUNDED)
t.add_column('Split', style='cyan')
t.add_column('ctx_len=0', justify='right')
t.add_column('resp empty/special-only', justify='right')
t.add_column('resp special-only', justify='right')
t.add_column('resp missing EOS', justify='right')
for split_name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    a, b, c, d = _check_suspicious(data, split_name)
    def _c(n): return f'[red]{n}[/red]' if n > 0 else '[green]0[/green]'
    t.add_row(split_name, _c(a), _c(b), _c(c), _c(d))
console.print(t)


## Section E — Sequence Length Statistics

In [ ]:
# === SECTION E.1: Length stats table — ctx and resp per split ===

def _lengths(data):
    ctx_lens  = [len(r.get('ctx', [])) for r in data]
    resp_lens = [len(r.get('resp', [])) for r in data]
    return np.array(ctx_lens), np.array(resp_lens)

all_lens = {}
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    cl, rl = _lengths(sdata)
    all_lens[sname] = (cl, rl)

def _stats(arr):
    if len(arr) == 0:
        return ['—'] * 6
    return [f'{arr.min()}', f'{arr.max()}', f'{arr.mean():.1f}',
            f'{np.median(arr):.0f}', f'{np.percentile(arr,95):.0f}', f'{np.percentile(arr,99):.0f}']

t = Table(title='Sequence Length Statistics', box=box.ROUNDED)
t.add_column('Split/Seq', style='cyan')
t.add_column('Min', justify='right')
t.add_column('Max', justify='right')
t.add_column('Mean', justify='right')
t.add_column('Median', justify='right')
t.add_column('p95', justify='right')
t.add_column('p99', justify='right')
for sname, (cl, rl) in all_lens.items():
    t.add_row(f'{sname}/ctx', *_stats(cl))
    t.add_row(f'{sname}/resp', *_stats(rl))
    t.add_row('', '', '', '', '', '', '')
console.print(t)

# Also as DataFrame
rows = []
for sname, (cl, rl) in all_lens.items():
    for seq_type, arr in [('ctx', cl), ('resp', rl)]:
        if len(arr):
            rows.append({'split': sname, 'seq': seq_type, 'min': arr.min(), 'max': arr.max(),
                         'mean': round(arr.mean(),1), 'median': int(np.median(arr)),
                         'p95': int(np.percentile(arr,95)), 'p99': int(np.percentile(arr,99))})
df_stats = pd.DataFrame(rows)
df_stats


In [ ]:
# === SECTION E.2: Overlaid length histograms (ctx + resp, splits as colors) ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
SPLIT_COLORS = {'train': '#4878CF', 'val': '#6ACC65', 'test': '#D65F5F'}

for ax, seq_type, max_len, title in [
    (axes[0], 'ctx',  MAX_CTX_TOKENS + 10, 'Context Length Distribution'),
    (axes[1], 'resp', MAX_RESP_TOKENS + 12, 'Response Length Distribution'),
]:
    for sname, (cl, rl) in all_lens.items():
        arr = cl if seq_type == 'ctx' else rl
        if len(arr) == 0: continue
        ax.hist(np.clip(arr, 0, max_len), bins=40, alpha=0.5,
                color=SPLIT_COLORS[sname], label=sname, edgecolor='none')
    ax.set_xlabel('Length (tokens)')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend()

fig.tight_layout()
_save_fig(fig, 'e2_length_histograms')
plt.show()


In [ ]:
# === SECTION E.3: Scatter ctx_len vs resp_len (5000-pt train sample, Pearson r) ===

from scipy import stats as scipy_stats

cl_train, rl_train = all_lens['train']
n_scatter = min(5000, len(cl_train))
idx = np.random.choice(len(cl_train), n_scatter, replace=False)
cx, ry = cl_train[idx], rl_train[idx]

r_val, p_val = scipy_stats.pearsonr(cx, ry)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(cx, ry, alpha=0.15, s=8, color='steelblue')
ax.set_xlabel('ctx length (tokens)')
ax.set_ylabel('resp length (tokens)')
ax.set_title(f'E.3 — ctx vs resp length (n={n_scatter:,}, r={r_val:.3f})')
m, b = np.polyfit(cx, ry, 1)
xline = np.linspace(cx.min(), cx.max(), 100)
ax.plot(xline, m*xline + b, 'r-', lw=1.5, alpha=0.8, label=f'slope={m:.3f}')
ax.legend()
fig.tight_layout()
_save_fig(fig, 'e3_ctx_resp_scatter')
plt.show()
console.print(f'Pearson r = {r_val:.4f}  (p = {p_val:.2e})')


In [ ]:
# === SECTION E.4: Truncation check ===

# ctx_len >= MAX_CTX_TOKENS means it hit the truncation limit
# resp_len >= MAX_RESP_TOKENS+2 means resp hit limit (includes SOS+EOS)
RESP_MAX_WITH_TOKENS = MAX_RESP_TOKENS + 2

t = Table(title=f'Truncation Check (ctx≥{MAX_CTX_TOKENS}, resp≥{RESP_MAX_WITH_TOKENS})', box=box.ROUNDED)
t.add_column('Split', style='cyan')
t.add_column('Total pairs', justify='right')
t.add_column('ctx at limit', justify='right')
t.add_column('ctx truncated %', justify='right')
t.add_column('resp at limit', justify='right')
t.add_column('resp truncated %', justify='right')
for sname, (cl, rl) in all_lens.items():
    if len(cl) == 0:
        t.add_row(sname, '0', '—', '—', '—', '—')
        continue
    n_ctx_trunc  = int((cl >= MAX_CTX_TOKENS).sum())
    n_resp_trunc = int((rl >= RESP_MAX_WITH_TOKENS).sum())
    t.add_row(sname, f'{len(cl):,}',
              f'{n_ctx_trunc:,}', f'{100*n_ctx_trunc/len(cl):.1f}%',
              f'{n_resp_trunc:,}', f'{100*n_resp_trunc/len(rl):.1f}%')
console.print(t)


In [ ]:
# === SECTION E.5: Response length distribution with median/p95 lines ===

fig, ax = plt.subplots(figsize=(9, 4))
rl = all_lens['train'][1]
ax.hist(rl, bins=50, color='#6ACC65', alpha=0.8, edgecolor='white', linewidth=0.4)
med = np.median(rl)
p95 = np.percentile(rl, 95)
ax.axvline(med, color='tomato', lw=2, ls='--', label=f'Median={med:.0f}')
ax.axvline(p95, color='orange', lw=2, ls=':', label=f'p95={p95:.0f}')
ax.axvline(RESP_MAX_WITH_TOKENS, color='gray', lw=1.5, ls='-', alpha=0.7,
           label=f'max_resp+2={RESP_MAX_WITH_TOKENS}')
ax.set_xlabel('Response Length (tokens, incl. SOS/EOS)')
ax.set_ylabel('Count')
ax.set_title('E.5 — Train Response Length Distribution')
ax.legend()
fig.tight_layout()
_save_fig(fig, 'e5_resp_len_dist')
plt.show()


## Section F — Token Distribution & Data Quality

In [ ]:
# === SECTION F.1: Top-30 most frequent tokens (train ctx+resp) ===

console.print('[bold]Building token frequency counter from train...[/bold]')
token_counter = Counter()
for rec in train_data:
    token_counter.update(rec.get('ctx', []))
    token_counter.update(rec.get('resp', []))

t = Table(title='Top-30 Most Frequent Tokens (train)', box=box.ROUNDED)
t.add_column('Rank', justify='right', style='dim')
t.add_column('ID', justify='right')
t.add_column('Piece', style='cyan')
t.add_column('Frequency', justify='right')
for rank, (tid, freq) in enumerate(token_counter.most_common(30), 1):
    piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else f'<id={tid}>'
    t.add_row(str(rank), str(tid), piece, f'{freq:,}')
console.print(t)


In [ ]:
# === SECTION F.2: UNK (id=1) occurrences per split ===

t = Table(title='UNK (id=1) Occurrences per Split', box=box.SIMPLE_HEAVY)
t.add_column('Split', style='cyan')
t.add_column('UNK count', justify='right')
t.add_column('Total tokens', justify='right')
t.add_column('UNK rate', justify='right')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    unk = sum(1 for r in sdata for seq in (r.get('ctx',[]), r.get('resp',[])) for tid in seq if tid == UNK_ID)
    tot = sum(len(r.get('ctx',[])) + len(r.get('resp',[])) for r in sdata)
    rate = unk/tot if tot > 0 else 0
    color = 'green' if rate < 0.001 else ('yellow' if rate < 0.01 else 'red')
    t.add_row(sname, f'{unk:,}', f'{tot:,}', f'[{color}]{rate:.5%}[/{color}]')
console.print(t)


In [ ]:
# === SECTION F.3: Type-Token Ratio (TTR) per split ===

t = Table(title='Type-Token Ratio (TTR) per Split', box=box.SIMPLE_HEAVY)
t.add_column('Split', style='cyan')
t.add_column('Unique types', justify='right')
t.add_column('Total tokens', justify='right')
t.add_column('TTR', justify='right')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    all_toks = [tid for r in sdata for seq in (r.get('ctx',[]), r.get('resp',[])) for tid in seq]
    n_types  = len(set(all_toks))
    n_tokens = len(all_toks)
    ttr = n_types / n_tokens if n_tokens > 0 else 0
    t.add_row(sname, f'{n_types:,}', f'{n_tokens:,}', f'{ttr:.4f}')
console.print(t)


In [ ]:
# === SECTION F.4: Length-ratio distribution (resp_len/ctx_len, train) ===

cl, rl = all_lens['train']
mask = cl > 0
ratios = rl[mask] / cl[mask]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(ratios, 0, 3), bins=50, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(np.median(ratios), color='tomato', lw=2, ls='--', label=f'Median={np.median(ratios):.3f}')
ax.set_xlabel('resp_len / ctx_len')
ax.set_ylabel('Count')
ax.set_title('F.4 — Response/Context Length Ratio (train, clipped at 3)')
ax.legend()
fig.tight_layout()
_save_fig(fig, 'f4_len_ratio')
plt.show()


In [ ]:
# === SECTION F.5: Duplicate detection in train (exact ctx+resp) ===

console.print('[bold]Checking for duplicate (ctx, resp) pairs in train...[/bold]')
seen_hashes = set()
n_dups = 0
for rec in train_data:
    key = (tuple(rec.get('ctx',[])), tuple(rec.get('resp',[])))
    h = hash(key)
    if h in seen_hashes:
        n_dups += 1
    else:
        seen_hashes.add(h)
color = 'green' if n_dups == 0 else 'red'
console.print(f'Duplicate pairs: [{color}]{n_dups:,}[/{color}] out of {len(train_data):,} train pairs '
              f'({100*n_dups/max(len(train_data),1):.3f}%)')
if n_dups == 0:
    console.print('[bold green]✓ No duplicates found — dedup working correctly.[/bold green]')
else:
    console.print('[bold red]✗ Duplicates found — check stage 4/6 dedup logic.[/bold red]')


In [ ]:
# === SECTION F.6: Short response check (< 5 content tokens) ===

t = Table(title='Short Response Check (< 5 content tokens)', box=box.SIMPLE_HEAVY)
t.add_column('Split', style='cyan')
t.add_column('Short (<5)', justify='right')
t.add_column('Total', justify='right')
t.add_column('%', justify='right')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    n_short = 0
    for rec in sdata:
        resp = rec.get('resp', [])
        content = [tid for tid in resp if tid not in (PAD_ID, SOS_ID, EOS_ID)]
        if len(content) < 5:
            n_short += 1
    pct = 100*n_short/max(len(sdata),1)
    color = 'green' if pct < 5 else ('yellow' if pct < 15 else 'red')
    t.add_row(sname, f'{n_short:,}', f'{len(sdata):,}', f'[{color}]{pct:.1f}%[/{color}]')
console.print(t)


In [ ]:
# === SECTION F.7: ctx sequences containing __eot__ + avg turns per ctx ===

t = Table(title='__eot__ Turn Structure in ctx', box=box.SIMPLE_HEAVY)
t.add_column('Split', style='cyan')
t.add_column('Has ≥1 __eot__', justify='right')
t.add_column('% with __eot__', justify='right')
t.add_column('Avg turns/ctx', justify='right')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    eot_counts = [rec.get('ctx', []).count(EOT_ID) for rec in sdata]
    n_has_eot = sum(1 for c in eot_counts if c > 0)
    avg_turns = (sum(c + 1 for c in eot_counts) / max(len(eot_counts), 1))  # turns = eot_count + 1
    pct_eot = 100 * n_has_eot / max(len(sdata), 1)
    t.add_row(sname, f'{n_has_eot:,}', f'{pct_eot:.1f}%', f'{avg_turns:.2f}')
console.print(t)


## Section G — Split Integrity Checks

In [ ]:
# === SECTION G.1: Cross-split source leakage (by ctx sequence hash) ===

console.print('[bold]Checking for cross-split leakage...[/bold]')
split_ctx_hashes = {}
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    split_ctx_hashes[sname] = {hash(tuple(r.get('ctx', []))) for r in sdata}

t = Table(title='Cross-Split Context Hash Leakage', box=box.ROUNDED)
t.add_column('Split A', style='cyan')
t.add_column('Split B', style='cyan')
t.add_column('Overlap', justify='right')
t.add_column('Status', justify='center')
pairs_checked = [('train','val'), ('train','test'), ('val','test')]
for sa, sb in pairs_checked:
    overlap = len(split_ctx_hashes[sa] & split_ctx_hashes[sb])
    status = '[bold green]PASS (0)[/bold green]' if overlap == 0 else f'[bold red]FAIL ({overlap:,})[/bold red]'
    t.add_row(sa, sb, str(overlap), status)
console.print(t)


In [ ]:
# === SECTION G.2: Token ID range check [0, VOCAB_SIZE) ===

t = Table(title=f'Token ID Range Check [0, {VOCAB_SIZE})', box=box.ROUNDED)
t.add_column('Split', style='cyan')
t.add_column('Min ID', justify='right')
t.add_column('Max ID', justify='right')
t.add_column('Out-of-range', justify='right')
t.add_column('Status', justify='center')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    all_ids = [tid for r in sdata for seq in (r.get('ctx',[]), r.get('resp',[])) for tid in seq]
    if not all_ids:
        t.add_row(sname, '—', '—', '—', '[yellow]EMPTY[/yellow]')
        continue
    min_id = min(all_ids)
    max_id = max(all_ids)
    oor    = sum(1 for tid in all_ids if tid < 0 or tid >= VOCAB_SIZE)
    status = '[bold green]PASS[/bold green]' if oor == 0 else f'[bold red]FAIL ({oor})[/bold red]'
    t.add_row(sname, str(min_id), str(max_id), str(oor), status)
console.print(t)


In [ ]:
# === SECTION G.3: SOS/EOS boundary verification ===

t = Table(title='SOS/EOS Boundary Verification', box=box.ROUNDED)
t.add_column('Split', style='cyan')
t.add_column('resp starts SOS', justify='right')
t.add_column('resp ends EOS', justify='right')
t.add_column('ctx has SOS/EOS', justify='right')
t.add_column('Status', justify='center')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    if not sdata:
        t.add_row(sname, '—', '—', '—', '[yellow]EMPTY[/yellow]')
        continue
    ok_sos   = sum(1 for r in sdata if r.get('resp', [SOS_ID])[0] == SOS_ID)
    ok_eos   = sum(1 for r in sdata if r.get('resp', [])[-1:] == [EOS_ID])
    ctx_leak = sum(1 for r in sdata if SOS_ID in r.get('ctx', []) or EOS_ID in r.get('ctx', []))
    n = len(sdata)
    s_sos   = f'{100*ok_sos/n:.1f}%'
    s_eos   = f'{100*ok_eos/n:.1f}%'
    c_leak  = str(ctx_leak)
    all_ok = (ok_sos == n and ok_eos == n and ctx_leak == 0)
    status = '[bold green]PASS[/bold green]' if all_ok else '[bold red]FAIL[/bold red]'
    t.add_row(sname, s_sos, s_eos, f'[{"red" if ctx_leak>0 else "green"}]{c_leak}[/{"red" if ctx_leak>0 else "green"}]', status)
console.print(t)


In [ ]:
# === SECTION G.4: __eot__ presence in ctx ===

t = Table(title='__eot__ Presence in ctx Sequences', box=box.ROUNDED)
t.add_column('Split', style='cyan')
t.add_column('ctx with __eot__', justify='right')
t.add_column('% with __eot__', justify='right')
t.add_column('Status', justify='center')
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    if not sdata:
        t.add_row(sname, '—', '—', '[yellow]EMPTY[/yellow]')
        continue
    n_has = sum(1 for r in sdata if EOT_ID in r.get('ctx', []))
    pct   = 100 * n_has / len(sdata)
    # Most ctx should have __eot__ (multi-turn context)
    # Single-turn dialogs (no eot) are acceptable but should be minority
    status = '[green]OK[/green]' if pct > 20 else '[yellow]WARN (low eot rate)[/yellow]'
    t.add_row(sname, f'{n_has:,}', f'{pct:.1f}%', status)
console.print(t)


In [ ]:
# === SECTION G.5: Final pass/fail checklist ===

checks = []
for sname, sdata in [('train', train_data), ('val', val_data), ('test', test_data)]:
    if not sdata:
        continue
    n = len(sdata)
    # G.1 leakage
    for other in ('val', 'test') if sname == 'train' else ('train',):
        if other in split_ctx_hashes:
            ov = len(split_ctx_hashes[sname] & split_ctx_hashes[other])
            checks.append((f'{sname}↔{other} leakage', ov == 0, f'{ov} overlap'))
    # G.3 SOS/EOS
    ok_sos = sum(1 for r in sdata if r.get('resp', [SOS_ID])[0] == SOS_ID)
    ok_eos = sum(1 for r in sdata if r.get('resp', [])[-1:] == [EOS_ID])
    ctx_leak = sum(1 for r in sdata if SOS_ID in r.get('ctx',[]) or EOS_ID in r.get('ctx',[]))
    checks.append((f'{sname} resp SOS', ok_sos == n, f'{ok_sos}/{n}'))
    checks.append((f'{sname} resp EOS', ok_eos == n, f'{ok_eos}/{n}'))
    checks.append((f'{sname} ctx no SOS/EOS', ctx_leak == 0, f'{ctx_leak} leaks'))

t = Table(title='G — Final Integrity Checklist', box=box.ROUNDED)
t.add_column('Check', style='cyan')
t.add_column('Detail')
t.add_column('Result', justify='center')
all_pass = True
for name, ok, detail in checks:
    result = '[bold green]PASS[/bold green]' if ok else '[bold red]FAIL[/bold red]'
    if not ok: all_pass = False
    t.add_row(name, detail, result)
console.print(t)
console.print(Panel(
    f'[bold {"green" if all_pass else "red"}]Overall: {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}[/bold {"green" if all_pass else "red"}]',
    border_style='green' if all_pass else 'red'
))


## Section H — Summary Dashboard

In [ ]:
# === SECTION H.1: Rich summary table ===

t = Table(title='Stage 6 Summary', box=box.ROUNDED)
t.add_column('Metric', style='cyan')
t.add_column('Train', justify='right')
t.add_column('Val', justify='right')
t.add_column('Test', justify='right')
rows_data = [
    ('Pairs', [f'{len(d):,}' for d in [train_data, val_data, test_data]]),
    ('Vocab size', [str(VOCAB_SIZE)]*3),
    ('Mean ctx len', [f'{all_lens[s][0].mean():.1f}' if len(all_lens[s][0]) else '—' for s in ["train","val","test"]]),
    ('Median ctx len', [f'{np.median(all_lens[s][0]):.0f}' if len(all_lens[s][0]) else '—' for s in ["train","val","test"]]),
    ('Mean resp len', [f'{all_lens[s][1].mean():.1f}' if len(all_lens[s][1]) else '—' for s in ["train","val","test"]]),
    ('Median resp len', [f'{np.median(all_lens[s][1]):.0f}' if len(all_lens[s][1]) else '—' for s in ["train","val","test"]]),
]
for name, vals in rows_data:
    t.add_row(name, *vals)
console.print(t)


In [ ]:
# === SECTION H.2: Multi-panel matplotlib dashboard ===

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Dataset sizes
ax1 = fig.add_subplot(gs[0, 0])
split_names = ['train', 'val', 'test']
sizes = [len(train_data), len(val_data), len(test_data)]
ax1.bar(split_names, sizes, color=['#4878CF','#6ACC65','#D65F5F'])
for i, v in enumerate(sizes):
    ax1.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
ax1.set_title('Dataset Sizes')
ax1.set_ylabel('Pairs')

# 2. ctx length histograms
ax2 = fig.add_subplot(gs[0, 1])
for sname, (cl, _), col in zip(split_names, [all_lens[s] for s in split_names], ['#4878CF','#6ACC65','#D65F5F']):
    if len(cl): ax2.hist(cl, bins=30, alpha=0.5, color=col, label=sname, density=True)
ax2.set_title('ctx Length Dist.')
ax2.set_xlabel('tokens')
ax2.legend(fontsize=8)

# 3. resp length histograms
ax3 = fig.add_subplot(gs[0, 2])
for sname, (_, rl), col in zip(split_names, [all_lens[s] for s in split_names], ['#4878CF','#6ACC65','#D65F5F']):
    if len(rl): ax3.hist(rl, bins=30, alpha=0.5, color=col, label=sname, density=True)
ax3.set_title('resp Length Dist.')
ax3.set_xlabel('tokens')
ax3.legend(fontsize=8)

# 4. Scatter ctx vs resp
ax4 = fig.add_subplot(gs[1, 0])
cl_tr, rl_tr = all_lens['train']
n_s = min(3000, len(cl_tr))
idx = np.random.choice(len(cl_tr), n_s, replace=False)
ax4.scatter(cl_tr[idx], rl_tr[idx], alpha=0.1, s=5, color='steelblue')
ax4.set_xlabel('ctx len')
ax4.set_ylabel('resp len')
ax4.set_title('ctx vs resp length')

# 5. Length ratio
ax5 = fig.add_subplot(gs[1, 1])
mask = cl_tr > 0
ratios_plot = rl_tr[mask] / cl_tr[mask]
ax5.hist(np.clip(ratios_plot, 0, 3), bins=40, color='steelblue', alpha=0.8)
ax5.axvline(np.median(ratios_plot), color='tomato', lw=1.5, ls='--')
ax5.set_xlabel('resp/ctx ratio')
ax5.set_title('Length Ratio (resp/ctx)')

# 6. Top-20 tokens
ax6 = fig.add_subplot(gs[1, 2])
top20_ids, top20_freqs = zip(*token_counter.most_common(20))
top20_pieces = [sp.id_to_piece(tid) if tid < sp.GetPieceSize() else str(tid) for tid in top20_ids]
ax6.barh(range(20), top20_freqs[::-1], color='steelblue', alpha=0.8)
ax6.set_yticks(range(20))
ax6.set_yticklabels([p[:15] for p in top20_pieces[::-1]], fontsize=8)
ax6.set_xlabel('Frequency')
ax6.set_title('Top-20 Tokens (train)')

# 7. Integrity pie (pass/fail)
ax7 = fig.add_subplot(gs[2, 0])
n_pass = sum(1 for _, ok, _ in checks if ok)
n_fail = len(checks) - n_pass
ax7.pie([n_pass, max(n_fail, 0)], labels=[f'Pass ({n_pass})', f'Fail ({n_fail})'],
        colors=['#6ACC65', '#D65F5F'], autopct='%1.0f%%', startangle=90)
ax7.set_title('Integrity Checks')

# 8. Response length dist (train only)
ax8 = fig.add_subplot(gs[2, 1])
ax8.hist(all_lens['train'][1], bins=50, color='#6ACC65', alpha=0.8)
ax8.axvline(np.median(all_lens['train'][1]), color='tomato', lw=1.5, ls='--', label='Median')
ax8.axvline(np.percentile(all_lens['train'][1], 95), color='orange', lw=1.5, ls=':', label='p95')
ax8.legend(fontsize=8)
ax8.set_title('Train resp Length')
ax8.set_xlabel('tokens')

# 9. Key stats text
ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
stats_text = (
    f'Vocab size: {VOCAB_SIZE:,}\n'
    f'Train pairs: {len(train_data):,}\n'
    f'Val pairs:   {len(val_data):,}\n'
    f'Test pairs:  {len(test_data):,}\n'
    f'Max ctx:     {MAX_CTX_TOKENS}\n'
    f'Max resp:    {MAX_RESP_TOKENS}\n'
    f'PAD=0  UNK=1\n'
    f'SOS=2  EOS=3\n'
    f'EOT_id={EOT_ID}\n'
    f'Integrity: {n_pass}/{len(checks)} pass'
)
ax9.text(0.05, 0.95, stats_text, transform=ax9.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.4))
ax9.set_title('Key Stats')

fig.suptitle('Stage 6 — Encoded Pairs Dashboard', fontsize=16, fontweight='bold')
_save_fig(fig, 'h2_dashboard')
plt.show()
